# EDA có câu hỏi: khách nào rời bỏ, và nên nhắm nhóm nào?

**Bối cảnh.** Head of Retention của một nhà mạng có ngân sách cho **một** chương trình giữ chân trong quý tới và
hỏi: *"Khách rời bỏ là ai? Nên nhắm vào nhóm nào để giảm nhiều khách rời nhất?"* Bạn có hai ngày.

Dữ liệu: bộ `telco-churn` (IBM, 7 043 khách, 21 cột). Một dòng là một khách tại thời điểm chụp dữ liệu; `Churn =
Yes` là khách đã rời trong tháng gần nhất. Các cột chính: `Contract` (month-to-month, một năm, hai năm), `tenure`
(số tháng đã gắn bó), `InternetService`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges` (tổng đã trả, kiểu
**chuỗi** trong tệp gốc), `SeniorCitizen` (0/1).

Đây không phải bài "vẽ hết mọi cột". Mỗi bước trả lời một câu hỏi con, theo quy trình ở phần 7 lý thuyết: hiểu
dữ liệu → chất lượng tối thiểu → bảng tỉ lệ theo nhóm → so phân phối → phân tầng → đề xuất. Cuối bài, **làm phần
trắc nghiệm trong app**: các câu hỏi là những con số bạn sẽ đưa vào đề xuất.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, vẽ và in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/07-descriptive-eda/exercises/intermediate/01-telco-churn-eda -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

raw = pd.read_csv(ensure("telco-churn") / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(raw.shape)
print(raw.dtypes.value_counts())
raw.head(3).T

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Hiểu dữ liệu và làm sạch tối thiểu

`TotalCharges` là chuỗi vì có ô chỉ chứa khoảng trắng. Trước khi đổi kiểu, hãy xem các ô đó là ai (cell explore
bên dưới): nếu chúng có chung một đặc điểm, đó là **thông tin**, không phải lỗi ngẫu nhiên.

`clean_telco(raw) -> DataFrame`: trả bản sao với

- `TotalCharges` kiểu `float`: chuỗi rỗng hoặc chỉ có khoảng trắng thành `NaN`; chuỗi số khác đổi bình thường;
  giá trị không phải số (ví dụ `"abc"`) phải gây lỗi, không được lặng lẽ thành NaN;
- `SeniorCitizen` thành chuỗi `"Yes"` / `"No"` (từ 1 / 0) để cùng kiểu với các cột Yes/No khác;
- cột mới `churned` kiểu `bool`, `True` khi `Churn == "Yes"`.

`add_tenure_band(df) -> DataFrame`: bản sao có thêm cột `tenure_band` (categorical có thứ tự) với 5 nhóm
`"0-6"`, `"7-12"`, `"13-24"`, `"25-48"`, `"49-72"` (tháng; `0` thuộc nhóm đầu, cận phải tính vào nhóm).

In [ ]:
# explore: ô TotalCharges trống là ai?
blank = raw.TotalCharges.str.strip().eq("")
print(blank.sum(), "ô trống")
raw.loc[blank, ["customerID", "tenure", "Contract", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
TENURE_BANDS = ["0-6", "7-12", "13-24", "25-48", "49-72"]


def clean_telco(raw: pd.DataFrame) -> pd.DataFrame:
    """Bản sao: TotalCharges float (ô trống → NaN), SeniorCitizen "Yes"/"No", thêm cột bool churned."""
    # TODO: df = raw.copy()
    # TODO: TotalCharges: strip, chuỗi rỗng → NaN, rồi pd.to_numeric (errors="raise")
    # TODO: SeniorCitizen 0/1 → "No"/"Yes"; churned = Churn == "Yes"
    raise NotImplementedError


def add_tenure_band(df: pd.DataFrame) -> pd.DataFrame:
    """Bản sao có cột tenure_band: 0-6, 7-12, 13-24, 25-48, 49-72 (categorical có thứ tự)."""
    # TODO: pd.cut(df["tenure"], [-1, 6, 12, 24, 48, 72], labels=TENURE_BANDS)
    raise NotImplementedError

In [ ]:
# explore: dữ liệu sạch
df = add_tenure_band(clean_telco(raw))
print(f"{len(df):,} khách, {df.churned.sum():,} rời bỏ ({df.churned.mean():.1%}); TotalCharges NaN: {df.TotalCharges.isna().sum()}")
df[["tenure", "tenure_band", "SeniorCitizen", "TotalCharges", "churned"]].head()

## 2. Bảng tỉ lệ theo nhóm: tỉ lệ cao chưa đủ, cần biết nhóm lớn cỡ nào

`rate_table(df, col, target="churned") -> DataFrame`: một dòng cho mỗi giá trị của `col` (index tên `col`), các cột
đúng thứ tự:

| Cột | Ý nghĩa |
| --- | --- |
| `n` | số khách của nhóm |
| `churned` | số khách rời bỏ của nhóm (`target` là cột bool) |
| `rate` | `churned / n` |
| `share_of_n` | tỉ trọng của nhóm trong tổng số khách |
| `share_of_churned` | tỉ trọng của nhóm trong tổng số khách rời bỏ |
| `lift` | `rate` chia tỉ lệ rời bỏ chung |

Sắp theo `rate` giảm dần. Với cột categorical (như `tenure_band`), chỉ giữ nhóm có khách (`observed=True`).

`share_of_churned` là cột Head of Retention cần nhất: một nhóm churn 60% nhưng chỉ có 1% khách không đáng một
chương trình riêng.

In [ ]:
def rate_table(df: pd.DataFrame, col: str, target: str = "churned") -> pd.DataFrame:
    """n, churned, rate, share_of_n, share_of_churned, lift theo từng giá trị của col; sắp rate giảm dần."""
    # TODO: g = df.groupby(col, observed=True)[target]
    # TODO: n = g.size(), churned = g.sum(); rồi các tỉ lệ
    raise NotImplementedError

In [ ]:
# explore: bảng tỉ lệ cho các biến ứng viên
pd.set_option("display.width", 160)
for col in ["Contract", "tenure_band", "InternetService", "PaymentMethod", "SeniorCitizen", "TechSupport", "gender"]:
    print(rate_table(df, col).round(3), end="\n\n")

In [ ]:
# explore: tỉ lệ churn và tỉ trọng trong số khách rời bỏ, theo hợp đồng và theo nhóm tenure
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, col in zip(axes, ["Contract", "tenure_band"]):
    t = rate_table(df, col)
    if col == "tenure_band":
        t = t.sort_index()
    ax.barh(t.index.astype(str), t["rate"], color="#4c78a8")
    for y, (r, s) in enumerate(zip(t["rate"], t["share_of_churned"])):
        ax.annotate(f"{r:.0%} churn · {s:.0%} số khách rời", (r, y), xytext=(3, 0), textcoords="offset points", va="center", fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(f"Tỉ lệ churn theo {col}", loc="left")
fig.tight_layout()
plt.show()

## 3. So phân phối biến số giữa khách rời bỏ và khách ở lại

`compare_numeric(df, col, target="churned") -> DataFrame`: index là hai giá trị của `target` (`False`, `True`,
tên index là `target`), các cột đúng thứ tự `n` (số giá trị không NaN), `mean`, `median`, `std` (mẫu), `iqr`
(p75 − p25, mặc định pandas), `skew`.

So **median và IQR**, không chỉ mean: phân phối của khách rời bỏ có thể lệch khác hẳn khách ở lại.

In [ ]:
def compare_numeric(df: pd.DataFrame, col: str, target: str = "churned") -> pd.DataFrame:
    """n, mean, median, std, iqr, skew của col cho từng giá trị của target."""
    # TODO: g = df.groupby(target)[col]; g.count(), g.mean(), ...
    raise NotImplementedError

In [ ]:
# explore: ba biến số
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    print(col)
    print(compare_numeric(df, col).round(2), end="\n\n")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    for flag, label in [(False, "ở lại"), (True, "rời bỏ")]:
        x = np.sort(df.loc[df.churned == flag, col].dropna())
        ax.plot(x, np.arange(1, len(x) + 1) / len(x), label=label)
    ax.set_title(f"ECDF {col}", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

## 4. Phân tầng: "khách chi ít thì bỏ đi"?

Bảng trên cho thấy khách rời bỏ có `TotalCharges` thấp hơn. Trước khi viết "khách chi ít thì dễ rời", nhớ rằng
`TotalCharges` xấp xỉ cước tháng nhân số tháng đã ở lại. So lại **trong từng nhóm tenure**.

In [ ]:
# explore: median TotalCharges và MonthlyCharges theo nhóm tenure
for col in ["TotalCharges", "MonthlyCharges"]:
    print(col)
    print(df.groupby(["tenure_band", "churned"], observed=True)[col].median().unstack().round(1), end="\n\n")

## 5. Từ bảng tới đề xuất

Một chương trình, một nhóm. Ứng viên phải **lớn** (nhiều khách rời bỏ để giữ) và **rõ** (dễ nhận diện để nhắm
đúng). Thử vài phân khúc kết hợp và so `n`, `rate`, `share_of_churned`.

In [ ]:
# explore: vài phân khúc ứng viên
segments = {
    "month-to-month + cáp quang": (df.Contract == "Month-to-month") & (df.InternetService == "Fiber optic"),
    "month-to-month + tenure ≤ 12": (df.Contract == "Month-to-month") & (df.tenure <= 12),
    "electronic check": df.PaymentMethod == "Electronic check",
    "người cao tuổi": df.SeniorCitizen == "Yes",
}
rows = {name: {"n": m.sum(), "rate": df.churned[m].mean(), "share_of_churned": df.churned[m].sum() / df.churned.sum()}
        for name, m in segments.items()}
pd.DataFrame(rows).T.round(3)

Viết ra (cho chính bạn, trước khi làm trắc nghiệm):

1. Headline một câu trả lời Head of Retention, có con số.
2. Nhóm bạn đề xuất nhắm, và vì sao không chọn nhóm có tỉ lệ churn cao nhất.
3. Điều chưa biết: bảng này cho thấy tương quan; cần gì để biết chương trình giữ chân có tác dụng?

Xong thì mở app, làm phần **trắc nghiệm** của bài.